# 实验 11 — dlt 配置与密钥管理

**目标**：理解 dlt 的三层配置体系，知道生产上怎么提供 API key、数据库连接、数据集名等参数，且**不把密钥写进代码**。

## dlt 的配置/密钥读取优先级

1. **环境变量**（最高优先级，CI/CD 最常用）：`SOURCES__FRANKFURTER__API_KEY=xxx`、`DESTINATION__POSTGRES__CREDENTIALS__PASSWORD=xxx`
2. **`.dlt/secrets.toml`** 和 **`.dlt/config.toml`**（本地开发）
3. 函数参数默认值（兜底）

非密钥（log_level、batch size、起始日期等）放 `config.toml`；密钥放 `secrets.toml`（必须 gitignore）。

参考：[.dlt/config.toml.example](../.dlt/config.toml.example) 与 [.dlt/secrets.toml.example](../.dlt/secrets.toml.example)。

In [ ]:
# 在 resource 参数上声明 dlt.secrets.value / dlt.config.value，dlt 会去三层位置找
import os, dlt, requests
from typing import Iterator

@dlt.resource(name='echo')
def echo(
    api_key: str = dlt.secrets.value,
    start_date: str = dlt.config.value,
) -> Iterator[dict]:
    yield {'api_key_first4': (api_key or '')[:4], 'start_date': start_date}

# 方式 1：用环境变量提供（最贴近 prod）
os.environ['SOURCES__ECHO__API_KEY'] = 'sk_test_ABCDEFG'
os.environ['SOURCES__ECHO__START_DATE'] = '2024-01-01'

pipe = dlt.pipeline(
    pipeline_name='secrets_demo',
    destination=dlt.destinations.duckdb('../data/sandbox.duckdb'),
    dataset_name='raw_secrets_demo',
)
print(pipe.run(echo()))

In [ ]:
# 看 dlt 实际拿到了什么
import duckdb
duckdb.connect('../data/sandbox.duckdb').execute(
    'select * from raw_secrets_demo.echo').df()

In [ ]:
# 方式 2：用 dlt.config / dlt.secrets 主动读
# 这种写法在你需要在 source 函数体里动态用配置时更直观
import dlt

# 也可以指定路径：dlt.config['sources.ecb_rates_pipeline.start_date']
try:
    print('SOURCES__ECHO__API_KEY     ->', dlt.secrets['sources.echo.api_key'])
    print('SOURCES__ECHO__START_DATE  ->', dlt.config['sources.echo.start_date'])
except Exception as e:
    print('miss:', e)

In [ ]:
# 移除环境变量、改用 .dlt/secrets.toml
for k in ['SOURCES__ECHO__API_KEY','SOURCES__ECHO__START_DATE']:
    os.environ.pop(k, None)

from pathlib import Path
secrets_dir = Path('../.dlt'); secrets_dir.mkdir(exist_ok=True)
(secrets_dir / 'secrets.toml').write_text('''[sources.echo]
api_key = "toml_secret_XYZ"
''')
(secrets_dir / 'config.toml').write_text('''[sources.echo]
start_date = "2024-06-01"
''')

# 重启 dlt 配置上下文，再跑
import importlib, dlt
importlib.reload(dlt)

@dlt.resource(name='echo')
def echo2(api_key: str = dlt.secrets.value, start_date: str = dlt.config.value):
    yield {'api_key_first4': (api_key or '')[:4], 'start_date': start_date}

pipe = dlt.pipeline(pipeline_name='secrets_demo',
                   destination=dlt.destinations.duckdb('../data/sandbox.duckdb'),
                   dataset_name='raw_secrets_demo')
print(pipe.run(echo2()))
import duckdb
print(duckdb.connect('../data/sandbox.duckdb').execute('select * from raw_secrets_demo.echo order by _dlt_load_id desc limit 2').df())

In [ ]:
# 清理实验产物（避免污染 git 仓库）
import os
for f in ['../.dlt/secrets.toml', '../.dlt/config.toml']:
    if os.path.exists(f): os.remove(f); print('removed', f)

## 生产环境踩坑提示

- **永远不要把 secrets.toml 提交 git**。`.gitignore` 里必须有 `.dlt/secrets.toml`（本仓库已配置）。
- **CI/CD 用环境变量**。把秘密放进 GitHub Actions secrets / Vault / AWS SM，注入成 `DESTINATION__SNOWFLAKE__CREDENTIALS__PASSWORD` 形式即可。
- **变量命名规则**：`<SECTION>__<KEY>` 双下划线分级，全大写。
- **配置的层级路径**：`sources.<source_name>.<key>` 或 `destination.<dest_name>.credentials.<key>`。命名空间错了 dlt 会安静地读不到，用 `dlt.secrets['...']` 取一次能快速验证。
- **千万别在 print 里打 api_key**。生产 logger 要 redact。

## 思考题

- 同时设置环境变量和 secrets.toml，谁赢？（试一下）
- 怎么给 DuckDB 之外的 destination（如 Snowflake）提供凭证？用什么环境变量名？
- 在你公司用 Vault / AWS Secrets Manager，你会怎么把秘密注入 dlt？